# SHAP

This version specifically fixes the issue where the six box plots of **a–f all display NA**, and adds stricter column name checking.

## Direct reason
The reason why the boxplots in panels a–f are all empty indicates that the two arrays passed into `boxplot()` are both empty arrays.
This type of problem is usually not caused by SHAP itself, but rather by **the drawing phase mistaking the feature abbreviation (such as DK, PR, UF) as the original data column name** to obtain the value, resulting in the fact that the numerical column in `df_region` is not actually obtained.

:

- There are both in `importance_table.csv`
  - `Feature_full`, for example `Distance_to_karst`
  - `Feature`, for example `DK`
- What really exists in the original data table is the **full name column**, such as `Distance_to_karst`, `Precip_hist_2000_2010_2020`
- Once you misuse `DK`, `PR`, `UF` to index the original table when drawing, you will get empty data, and then the top average can only display `NA`

## Repair strategy
1. **Force to use `Feature_full` first**
2. If missing, automatically fall back to the mapping through `ABBR_TO_FULL`
3. If the column still cannot be obtained, **report an error directly** and no longer draw an empty image silently
4. Box plot data are uniformly converted to numerical type to prevent string type data from being mixed in
5. Added diagnostic printing to quickly locate which area or feature has the problem.

In [ ]:
# =========================================================
# 0) Import library
# =========================================================
import os
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:

# =========================================================
# 1) Path
# =========================================================
home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
data_path = os.path.join(base_path, "data")
path_name = "Positive_Negative_balanced"
print(path_name)

out_dir = os.path.join(base_path, "outputs", path_name, "shap", "division")
print(f"out_dir:{out_dir}")
os.makedirs(out_dir, exist_ok=True)

df_path = os.path.join(
    data_path, "Extracted_HAVE_future", "Positive_Negative_balanced",
    "AllFeatures_Positive_Negative_balanced_25366_ssp1_cleaned_division.csv"
)
if not os.path.exists(df_path):
    alt = os.path.join(os.getcwd(), "AllFeatures_Positive_Negative_balanced_25366_ssp1_cleaned_division.csv")
    if os.path.exists(alt):
        df_path = alt

df = pd.read_csv(df_path)
print("Loaded:", df.shape, "from", df_path)

DATA_CSV = df_path
OUT_DIR = out_dir
os.makedirs(OUT_DIR, exist_ok=True)

FAST_MODE = False
USE_12_FEATURES = True

In [ ]:
# =========================================================
# 2)
# =========================================================
df = pd.read_csv(DATA_CSV)
print("Loaded:", df.shape)
print("Columns:", len(df.columns))

TARGET_COL = "Disaster"
DIV_COL = "DIV_EN"

required_basic = [TARGET_COL, DIV_COL]
for c in required_basic:
    if c not in df.columns:
        raise ValueError(f"The data is missing a required column:{c}")

In [ ]:
# =========================================================
# 3) Feature definition
# =========================================================
FEATURES_10 = [
    "Distance_to_karst",
    "Depth_to_Bedrock",
    "Distance_to_Fault_m",
    "UrbanFrac_hist_2000_2010_2020",
    "PopTotal_hist_2000_2010_2020",
    "ImperviousIndex_hist_2000_2010_2020",
    "LAI_hist_2000_2010_2020",
    "Precip_hist_2000_2010_2020",
    "WTD_hist_2000_2010_2020",
    "HDS_hist_2000_2010_2020",
]

FEATURES_12 = [
    "Distance_to_karst",
    "Depth_to_Bedrock",
    "Distance_to_Fault_m",
    "UrbanFrac_hist_2000_2010_2020",
    "PopTotal_hist_2000_2010_2020",
    "ImperviousIndex_hist_2000_2010_2020",
    "LAI_hist_2000_2010_2020",
    "Precip_hist_2000_2010_2020",
    "Tas_hist_2000_2010_2020",
    "Huss_hist_2000_2010_2020",
    "WTD_hist_2000_2010_2020",
    "HDS_hist_2000_2010_2020",
]

FEATURES = FEATURES_12 if USE_12_FEATURES else FEATURES_10

ABBR = {
    "Distance_to_karst": "DK",
    "Depth_to_Bedrock": "DB",
    "Distance_to_Fault_m": "DF",
    "UrbanFrac_hist_2000_2010_2020": "UF",
    "PopTotal_hist_2000_2010_2020": "PT",
    "ImperviousIndex_hist_2000_2010_2020": "IP",
    "LAI_hist_2000_2010_2020": "LAI",
    "Precip_hist_2000_2010_2020": "PR",
    "Tas_hist_2000_2010_2020": "TAS",
    "Huss_hist_2000_2010_2020": "HUSS",
    "WTD_hist_2000_2010_2020": "WTD",
    "HDS_hist_2000_2010_2020": "HDS",
}

FULL_TO_ABBR = {k: v for k, v in ABBR.items()}
ABBR_TO_FULL = {v: k for k, v in ABBR.items()}

missing_features = [c for c in FEATURES if c not in df.columns]
if missing_features:
    raise ValueError(
        "The following feature columns do not exist in the data, please check whether you are currently using the 10 feature version or the 12 feature version: \\n"
        + "\n".join(missing_features)
    )

print("Using features:", FEATURES)

In [ ]:
# =========================================================
# 4) Factor grouping
# =========================================================
if USE_12_FEATURES:
    G1 = "Anthro"
    G2 = "Climate"
    G3 = "Hydrogeo"

    GROUPS_BY_ABBR = {
        "UF":   [G1],
        "IP":   [G1],
        "PT":   [G1],
        "WTD":  [G1],
        "LAI":  [G1],
        "PR":   [G2],
        "TAS":  [G2],
        "HUSS": [G2],
        "DK":   [G3],
        "DB":   [G3],
        "DF":   [G3],
        "HDS":  [G3],
    }
else:
    G1 = "Anthro"
    G2 = "Hydro\nEnv"
    G3 = "Geo"

    GROUPS_BY_ABBR = {
        "UF":  [G1],
        "PT":  [G1],
        "IP":  [G1, G2],
        "WTD": [G1, G2],
        "HDS": [G1, G2],
        "PR":  [G2],
        "LAI": [G2],
        "DK":  [G3],
        "DB":  [G3],
        "DF":  [G3],
    }

COLOR_INC = "#C6464A"
COLOR_DEC = "#2F5A73"

COL_DARK_RED = "#7A0019"
COL_RED      = "#C1122F"
COL_BEIGE    = "#F3E6C6"
COL_LBLUE    = "#6FA6C8"
COL_DBLUE    = "#0B3D5C"

GROUP_COLOR = {
    G1: COL_DARK_RED,
    G2: COL_RED,
    G3: COL_BEIGE,
    "Mixed": COL_LBLUE,
    "Other": COL_DBLUE,
}

In [ ]:
# =========================================================
# 5) Area list
# =========================================================
REGIONS_7 = [
    "Northeast China",
    "Northwest China",
    "North China",
    "Central China",
    "Southwest China",
    "South China",
    "East China",
]
REGIONS = REGIONS_7 + ["National"]

In [ ]:
# =========================================================
# 6) Tool function
# =========================================================
def safe_name(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_\-]+", "", s)
    return s

def save_png_svg(fig, out_folder, stem, dpi=300):
    os.makedirs(out_folder, exist_ok=True)
    fig.savefig(os.path.join(out_folder, f"{stem}.png"), dpi=dpi, bbox_inches="tight")
    fig.savefig(os.path.join(out_folder, f"{stem}.svg"), bbox_inches="tight")
    plt.close(fig)

def groups_for_abbr(a: str):
    return GROUPS_BY_ABBR.get(a, [])

def bar_color_for_groups(groups):
    if len(groups) == 1:
        return GROUP_COLOR.get(groups[0], GROUP_COLOR["Other"])
    if len(groups) >= 2:
        return GROUP_COLOR["Mixed"]
    return GROUP_COLOR["Other"]

def style_axes_box(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1)
    ax.spines["bottom"].set_linewidth(1)
    ax.tick_params(axis="both", which="major", length=3, width=1)

def style_axes_bar(ax):
    for sp in ax.spines.values():
        sp.set_visible(True)
        sp.set_linewidth(1)
    ax.tick_params(axis="both", which="major", length=3, width=1)

def add_panel_letter(ax, letter, x=-0.18, y=1.04, fontsize=13):
    ax.text(x, y, letter, transform=ax.transAxes, fontweight="bold", fontsize=fontsize)

def get_feature_column_name(df_region: pd.DataFrame, feature_full: str, feature_abbr: str) -> str:
    '''The core functions of this repair:
    1. Use Feature_full first
    2. If Feature_full is missing, try to search the full name from the abbreviation.
    3. If neither can be found, an error will be reported directly.
    
The input must be a one-dimensional array. Current length ='''
    if pd.notna(feature_full) and feature_full in df_region.columns:
        return feature_full

    if pd.notna(feature_abbr):
        mapped = ABBR_TO_FULL.get(feature_abbr)
        if mapped is not None and mapped in df_region.columns:
            print(f"[Warn] Use short name{feature_abbr}Fallback mapping to original column{mapped}")
            return mapped

    raise KeyError(
        f".Feature_full={feature_full}, Feature={feature_abbr}"
    )

def get_numeric_group_arrays(df_region: pd.DataFrame, feat_col: str):
    '''Uniformly convert box plot data to numerical values and remove null values.'''
    inc = pd.to_numeric(
        df_region.loc[df_region[TARGET_COL] == 1, feat_col], errors="coerce"
    ).dropna().to_numpy()

    dec = pd.to_numeric(
        df_region.loc[df_region[TARGET_COL] == 0, feat_col], errors="coerce"
    ).dropna().to_numpy()

    return inc, dec

def boxplot_two_groups(ax, data_inc, data_dec, ylabel):
    b = ax.boxplot(
        [data_inc, data_dec],
        tick_labels=["Increased", "Decreased"],
        patch_artist=True,
        showfliers=False,
        showmeans=True,
        widths=0.75,
        meanprops=dict(marker="x", markeredgecolor="black", markerfacecolor="black", markersize=4),
        medianprops=dict(color="black", linewidth=1),
        whiskerprops=dict(color="black", linewidth=1),
        capprops=dict(color="black", linewidth=1),
    )
    for patch, c in zip(b["boxes"], [COLOR_INC, COLOR_DEC]):
        patch.set_facecolor(c)
        patch.set_edgecolor("black")
        patch.set_linewidth(0.8)

    ax.set_ylabel(ylabel)
    style_axes_box(ax)

    m1 = np.mean(data_inc) if len(data_inc) else np.nan
    m2 = np.mean(data_dec) if len(data_dec) else np.nan
    y0, y1 = ax.get_ylim()
    yr = (y1 - y0) if y1 > y0 else 1.0
    ax.set_ylim(y0, y1 + 0.18 * yr)
    ax.text(1, y1 + 0.04 * yr, f"{m1:.1f}" if np.isfinite(m1) else "NA", ha="center", va="bottom")
    ax.text(2, y1 + 0.04 * yr, f"{m2:.1f}" if np.isfinite(m2) else "NA", ha="center", va="bottom")

In [ ]:
# =========================================================
# 7) Model training and SHAP
# =========================================================
def train_rf_classifier(X, y, random_state=42, fast_mode=True):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=random_state, stratify=y
    )

    if fast_mode:
        model = RandomForestClassifier(
            n_estimators=300,
            max_depth=20,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        )
        model.fit(X_train, y_train)
        best = model
        best_params = {
            "n_estimators": 300,
            "max_depth": 20,
            "min_samples_leaf": 2,
        }
    else:
        rf = RandomForestClassifier(
            random_state=random_state, n_jobs=-1, class_weight="balanced"
        )
        param_grid = {
            "n_estimators": [200, 500],
            "max_depth": [None, 20],
            "min_samples_leaf": [1, 2],
        }
        gs = GridSearchCV(rf, param_grid=param_grid, cv=3, n_jobs=-1, scoring="roc_auc")
        gs.fit(X_train, y_train)
        best = gs.best_estimator_
        best_params = gs.best_params_

    prob = best.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)

    metrics = {
        "AUC": roc_auc_score(y_test, prob),
        "ACC": accuracy_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "best_params": best_params,
    }
    return best, X_train, X_test, y_test, metrics

def compute_shap_importance(model, X_background, X_test):
    import shap

    if len(X_background) > 300:
        bg = shap.sample(X_background, 300, random_state=42)
    else:
        bg = X_background

    explainer = shap.Explainer(model, bg, algorithm="tree")
    shap_exp = explainer(X_test, check_additivity=False)

    vals = np.array(shap_exp.values)

    if vals.ndim == 3 and vals.shape[-1] == 2:
        vals_pos = vals[..., 1]
    else:
        vals_pos = vals

    mean_abs = np.abs(vals_pos).mean(axis=0)

    signs = []
    for j in range(X_test.shape[1]):
        xj = X_test.iloc[:, j].values
        sj = vals_pos[:, j]
        if np.all(np.isfinite(xj)) and np.all(np.isfinite(sj)) and np.std(xj) > 0 and np.std(sj) > 0:
            r = np.corrcoef(xj, sj)[0, 1]
        else:
            r = np.nan
        signs.append("+" if (np.isfinite(r) and r > 0) else "-")

    return vals_pos, mean_abs, signs

In [ ]:
# =========================================================
# 8) Each area outputs 1 whole image
# =========================================================
def run_one_region(df_all, region_name: str):
    if region_name == "National":
        df_region = df_all.loc[df_all[DIV_COL].isin(REGIONS_7)].copy()
    else:
        df_region = df_all.loc[df_all[DIV_COL] == region_name].copy()

    use_cols = [DIV_COL, TARGET_COL] + FEATURES
    for c in use_cols:
        if c not in df_region.columns:
            raise ValueError(f"{region_name}Missing column:{c}")

    df_region = df_region[use_cols].replace([np.inf, -np.inf], np.nan).dropna()

    if df_region.empty:
        print(f"[Skip] {region_name}: No data after cleaning.")
        return

    if df_region[TARGET_COL].nunique() < 2:
        print(f"[Skip] {region_name}: Disaster has only one category.")
        return

    region_tag = safe_name(region_name)
    region_out = os.path.join(OUT_DIR, region_tag)
    os.makedirs(region_out, exist_ok=True)

    X = df_region[FEATURES].copy()
    y = df_region[TARGET_COL].astype(int).copy()

    model, X_train, X_test, y_test, metrics = train_rf_classifier(
        X, y, random_state=42, fast_mode=FAST_MODE
    )
    print(f"\n[{region_name}] n={len(df_region)} metrics:", metrics)

    sv_pos, mean_abs, signs = compute_shap_importance(model, X_train, X_test)

    imp = pd.DataFrame({
        "Feature_full": FEATURES,
        "Feature": [FULL_TO_ABBR[f] for f in FEATURES],
        "Mean_SHAP": mean_abs,
        "Sign": signs,
    })
    imp["Groups"] = imp["Feature"].apply(groups_for_abbr)
    imp["Groups_str"] = imp["Groups"].apply(lambda lst: ";".join(lst))
    imp = imp.sort_values("Mean_SHAP", ascending=False).reset_index(drop=True)

    imp.to_csv(
        os.path.join(region_out, f"{region_tag}_importance_table.csv"),
        index=False, encoding="utf-8-sig"
    )

    top6 = imp.head(6).copy()

    fig = plt.figure(figsize=(7.1, 6.8))
    gs = GridSpec(
        nrows=3, ncols=3,
        height_ratios=[1, 1, 1.35],
        hspace=0.45, wspace=0.55, figure=fig
    )

    letters = list("abcdef")
    for i in range(6):
        r = 0 if i < 3 else 1
        c = i % 3
        ax = fig.add_subplot(gs[r, c])

        feat_full = top6.loc[i, "Feature_full"]
        feat_abbr = top6.loc[i, "Feature"]

        feat_col = get_feature_column_name(df_region, feat_full, feat_abbr)
        inc, dec = get_numeric_group_arrays(df_region, feat_col)

        print(
            f"[Diag] {region_name} | panel {letters[i]} | "
            f"abbr={feat_abbr} | full={feat_full} | col={feat_col} | "
            f"n_inc={len(inc)} | n_dec={len(dec)} | "
            f"mean_inc={np.mean(inc) if len(inc) else np.nan:.4f} | "
            f"mean_dec={np.mean(dec) if len(dec) else np.nan:.4f}"
        )

        if len(inc) == 0 or len(dec) == 0:
            raise ValueError(
                f"{region_name}."
                f" panel={letters[i]}, feat_abbr={feat_abbr}, feat_full={feat_full}, feat_col={feat_col}"
            )

        boxplot_two_groups(ax, inc, dec, ylabel=feat_abbr)
        add_panel_letter(ax, letters[i], x=-0.18, y=1.04, fontsize=13)

    axg = fig.add_subplot(gs[2, :])
    add_panel_letter(axg, "g", x=-0.10, y=1.02, fontsize=13)

    plot_df = imp.copy().sort_values("Mean_SHAP", ascending=False)
    x = np.arange(len(plot_df))
    colors = [bar_color_for_groups(g) for g in plot_df["Groups"].values]

    bars = axg.bar(x, plot_df["Mean_SHAP"].values, width=0.55)
    for rect, col in zip(bars, colors):
        rect.set_facecolor(col)
        rect.set_edgecolor("black")
        rect.set_linewidth(0.6)

    axg.set_ylabel("Mean |SHAP| value")
    axg.set_xticks(x)
    axg.set_xticklabels(plot_df["Feature"].values, rotation=90, ha="center")
    style_axes_bar(axg)

    ymax = float(plot_df["Mean_SHAP"].max()) if len(plot_df) else 1.0
    for i, (v, sgn) in enumerate(zip(plot_df["Mean_SHAP"].values, plot_df["Sign"].values)):
        axg.text(i, v + 0.03 * ymax, sgn, ha="center", va="bottom")

    axg.set_ylim(0, ymax * 1.60)
    axg.text(0.10, 0.95, f"AUC={metrics['AUC']:.2f}",
             transform=axg.transAxes, ha="left", va="top")

    axh = axg.inset_axes([0.62, 0.55, 0.34, 0.38])

    group_order = [G1, G2, G3]
    group_means = []
    for g in group_order:
        mask = plot_df["Groups"].apply(lambda lst: g in lst)
        group_means.append(plot_df.loc[mask, "Mean_SHAP"].mean() if mask.any() else 0.0)

    xi = np.arange(len(group_order))
    b2 = axh.bar(xi, group_means, width=0.45)
    for rect, g in zip(b2, group_order):
        rect.set_facecolor(GROUP_COLOR[g])
        rect.set_edgecolor("black")
        rect.set_linewidth(0.6)

    axh.set_xticks(xi)
    axh.set_xticklabels(group_order, rotation=90, ha="center")
    axh.tick_params(axis="both", which="major", length=2, width=0.8)
    for sp in axh.spines.values():
        sp.set_linewidth(1)

    fig.suptitle(region_name, y=0.995, fontweight="bold")

    stem = f"{region_tag}_Fig_Box_SHAP_7panels"
    save_png_svg(fig, region_out, stem)

In [ ]:
# =========================================================
# 9) Main program
# =========================================================
def main():
    for r in REGIONS:
        run_one_region(df, r)
    print("\nDone. Outputs saved under:", OUT_DIR)

main()

## Two points you need to pay attention to

### 1. The nature of NA this time is not that the SHAP value is NA.
g The graph can be drawn normally, indicating that the model training and SHAP sorting itself are successful.
The real error is that when **a–f retrieves the feature distribution from the original table, the column names do not match**.

### 2. The ipynb you are currently uploading is still the old version 10 features
That is, it has not yet incorporated `TAS` and `HUSS` into `FEATURES`.
If what you actually want to draw now is the 12-feature version, change the above configuration to:
To switch to the 12-feature version, use the following configuration:

```python
USE_12_FEATURES = True
```

to switch to the new version.